# 2.4 — Apply Cortex Functions in Data Pipelines

**Exam domain:** Gen AI Functions (Domain 2.0) · **Weight:** 38%

## The problem this solves

You wrote the enrichment query in 2.1 and it works. Then someone runs it again on Monday, over the whole
table, including the 200,000 rows that were already enriched last week. The bill arrives and nobody can
explain it.

Batch AI over a growing table has one hard requirement: process each row exactly once. Snowflake's answer
is a change stream to say what is new, and a scheduled task to act on it — the same two objects you would
use for any incremental pipeline, with a much more expensive payload in the middle.

## What you will be able to do

- Build a stream-driven pipeline that calls AI functions only on new rows
- Order enrichment stages so personal data never reaches a second model
- Schedule the work with a task that does not start a warehouse when there is nothing to do
- Read stream metadata columns to tell inserts apart from the insert half of an update
- Name the cost drivers in an AI pipeline and the lever for each one

## Before you start

- Run `setup/dataset.sql`. It creates `SUPPORT_TICKETS` and `SUPPORT_TICKETS_STREAM`.
- The owning role needs `USE AI FUNCTIONS` plus `SNOWFLAKE.CORTEX_USER`, and `EXECUTE TASK` on the
  account to run tasks.
- Notebook 2.1 covers the AI functions used in the enrichment step.

📖 **Snowflake documentation for this notebook**
- [Change tracking using table streams](https://docs.snowflake.com/en/user-guide/streams-intro)
- [CREATE TASK](https://docs.snowflake.com/en/sql-reference/sql/create-task)
- [AI_CLASSIFY](https://docs.snowflake.com/en/sql-reference/functions/ai_classify)
- [AI_SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_sentiment)
- [AI_REDACT](https://docs.snowflake.com/en/sql-reference/functions/ai_redact)
- [Cortex AI function cost management](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-func-cost-management)


---

## The shape of an AI pipeline

```
Source table  ->  STREAM (change capture)  ->  TASK  ->  AI enrichment  ->  Target table
```

Nothing here is specific to AI. What is specific to AI is that the middle step costs real money per row,
so the parts of the pattern that exist to avoid redundant work stop being tidiness and start being the
whole point.

| Stage | What Cortex does here |
|---|---|
| **Extraction** | `AI_PARSE_DOCUMENT`, `AI_EXTRACT` — pull structure out of raw text or files |
| **Enrichment** | `AI_CLASSIFY`, `AI_SENTIMENT`, `AI_TRANSLATE` — add derived columns |
| **Augmentation** | `AI_COMPLETE`, `AI_EMBED` — generate new content or vectors |
| **Transformation** | `AI_REDACT`, `SNOWFLAKE.CORTEX.SUMMARIZE` — reshape and clean for downstream consumers |

### Order matters more than it looks

Redaction goes first. Not because the pipeline breaks otherwise, but because every AI function after it
is another system that has now seen a customer's name. Putting `AI_REDACT` at the top means the only
thing that ever saw the raw text is the redaction call itself.

The second ordering decision is cheaper: filter before you enrich. A `WHERE language = 'en'` costs
nothing and removes rows you were about to spend money on.

→ [More on AI_REDACT](https://docs.snowflake.com/en/sql-reference/functions/ai_redact)


In [ ]:
%%sql
-- Target table for the pipeline
CREATE OR REPLACE TABLE GENAI_STUDY.PUBLIC.TICKETS_ENRICHED (
    ticket_id        NUMBER        PRIMARY KEY,
    created_at       TIMESTAMP_NTZ,
    category         VARCHAR(50),
    safe_text        VARCHAR(4000),       -- AI_REDACT output
    predicted_cat    VARCHAR(50),         -- AI_CLASSIFY :labels[0]
    sentiment        VARCHAR(20),         -- AI_SENTIMENT overall sentiment LABEL (not a score)
    english_text     VARCHAR(4000),       -- AI_TRANSLATE output
    summary          VARCHAR(500),        -- SNOWFLAKE.CORTEX.SUMMARIZE output
    embedding        VECTOR(FLOAT, 768),  -- AI_EMBED (arctic-embed-m-v1.5 = 768 dims)
    processed_at     TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);
-- AI_SENTIMENT returns labels, not numeric scores, so the sentiment column is a VARCHAR.

-- Streams require change tracking on the source
ALTER TABLE GENAI_STUDY.PUBLIC.SUPPORT_TICKETS SET CHANGE_TRACKING = TRUE;


In [ ]:
%%sql -r pipeline_setup_2
SHOW STREAMS LIKE 'SUPPORT_TICKETS_STREAM' IN SCHEMA GENAI_STUDY.PUBLIC;


---

## Streams: what changed since last time

A *stream* is not a copy of your data. It is an offset — a bookmark into the table's change history —
plus a set of metadata columns that describe each change. Querying it shows you the rows that have changed
since the bookmark.

| Column | What it holds |
|---|---|
| `METADATA$ACTION` | `INSERT` or `DELETE` |
| `METADATA$ISUPDATE` | `TRUE` when this row is one half of an `UPDATE`, which appears as a paired `DELETE` and `INSERT` |
| `METADATA$ROW_ID` | A stable, immutable row identifier for tracking a row across changes |

That pairing is why the enrichment queries below filter on
`METADATA$ACTION = 'INSERT' AND METADATA$ISUPDATE = FALSE`. Without the second condition, every update to
an existing ticket looks like a brand-new row and gets enriched again — which is both a duplicate in the
target table and a bill.

### The rule that catches people out

**A stream advances its offset only when it is consumed by a DML statement.** Selecting from it — as the
next cell does — shows you the rows and changes nothing. The `INSERT ... SELECT ... FROM stream` is what
moves the bookmark forward, and after that the same rows are gone.

Within a single transaction, repeated reads of a stream see the same set of records, so a multi-statement
transaction is consistent. Across transactions, once a DML statement has consumed the stream, the next
read starts from the new offset.

Change tracking has to be on for the source object. Creating a stream on a table enables it, and you can
set it explicitly with `ALTER TABLE ... SET CHANGE_TRACKING = TRUE`.

→ [More on streams](https://docs.snowflake.com/en/user-guide/streams-intro)


In [ ]:
%%sql
-- Manual run of the enrichment pipeline: every stage in one INSERT ... SELECT.
INSERT INTO GENAI_STUDY.PUBLIC.TICKETS_ENRICHED
    (ticket_id, created_at, category, safe_text, predicted_cat,
     sentiment, english_text, summary, embedding)
SELECT
    ticket_id,
    created_at,
    category,
    -- Stage 1: transformation — strip PII first, so no PII reaches any other model
    AI_REDACT(ticket_text)                                                      AS safe_text,
    -- Stage 2: enrichment
    AI_CLASSIFY(ticket_text,
        ['billing','technical','shipping','feedback']):labels[0]::VARCHAR       AS predicted_cat,
    (SELECT c.value:sentiment::VARCHAR
       FROM LATERAL FLATTEN(AI_SENTIMENT(ticket_text):categories) c
      WHERE c.value:name::VARCHAR = 'overall')                                  AS sentiment,
    -- Stage 3: augmentation
    CASE WHEN language = 'en' THEN ticket_text
         ELSE AI_TRANSLATE(ticket_text, '', 'en') END                           AS english_text,
    SNOWFLAKE.CORTEX.SUMMARIZE(ticket_text)                                     AS summary,
    -- Stage 4: vectorisation
    AI_EMBED('snowflake-arctic-embed-m-v1.5', ticket_text)                      AS embedding
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS;


In [ ]:
%%sql -r pipeline_run_2
SELECT COUNT(*) AS rows_enriched FROM GENAI_STUDY.PUBLIC.TICKETS_ENRICHED;

-- Cost note: this calls several AI functions per row. Filter first, truncate long text, and
-- process only new rows (the stream pattern below) rather than re-running over history.


> ### ⚠️ Common misconceptions
>
> **"Querying the stream in a notebook to check it consumes the rows."**
> A plain `SELECT` does not advance the offset. Only a DML statement — `INSERT`, `MERGE`, `CREATE TABLE
> AS SELECT`, `COPY INTO <location>` — consumes the stream. This is good news for debugging, and it is
> also why a pipeline that "ran" but used a `SELECT` instead of an `INSERT` reprocesses the same rows
> forever.
> → [Streams](https://docs.snowflake.com/en/user-guide/streams-intro)
>
> **"`METADATA$ACTION = 'INSERT'` gives me the new rows."**
> It gives you new rows *and* the insert half of every update, because an update is recorded as a paired
> `DELETE` and `INSERT` with `METADATA$ISUPDATE = TRUE`. Omitting the `METADATA$ISUPDATE = FALSE`
> condition means re-running every AI function on rows you already enriched, and inserting a duplicate.
> → [Streams](https://docs.snowflake.com/en/user-guide/streams-intro)
>
> **"The enrichment query is one statement, so it is one AI call per row."**
> It is one call per AI function per row. The pipeline below calls five, so 10,000 new tickets are 50,000
> model invocations. Writing the same function twice — once in the `SELECT` list and once in a `WHERE`
> clause — doubles its share again.
> → [Cortex AI function cost management](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-func-cost-management)


In [ ]:
%%sql -r stream_processing_1
-- Stream-driven incremental processing: pay for AI only on new rows.
-- A stream's offset advances only when the stream is used in a DML statement --
-- this SELECT inspects it without consuming it.
SELECT SYSTEM$STREAM_HAS_DATA('GENAI_STUDY.PUBLIC.SUPPORT_TICKETS_STREAM') AS has_new_data;


In [ ]:
%%sql
INSERT INTO GENAI_STUDY.PUBLIC.TICKETS_ENRICHED
    (ticket_id, created_at, category, safe_text, predicted_cat, sentiment, english_text, summary, embedding)
SELECT
    s.ticket_id,
    s.created_at,
    s.category,
    AI_REDACT(s.ticket_text),
    AI_CLASSIFY(s.ticket_text, ['billing','technical','shipping','feedback']):labels[0]::VARCHAR,
    (SELECT c.value:sentiment::VARCHAR
       FROM LATERAL FLATTEN(AI_SENTIMENT(s.ticket_text):categories) c
      WHERE c.value:name::VARCHAR = 'overall'),
    CASE WHEN s.language = 'en' THEN s.ticket_text
         ELSE AI_TRANSLATE(s.ticket_text, '', 'en') END,
    SNOWFLAKE.CORTEX.SUMMARIZE(s.ticket_text),
    AI_EMBED('snowflake-arctic-embed-m-v1.5', s.ticket_text)
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS_STREAM s
WHERE s.METADATA$ACTION = 'INSERT'
  AND s.METADATA$ISUPDATE = FALSE;   -- exclude the INSERT half of an UPDATE pair
-- Stream metadata columns: METADATA$ACTION (INSERT | DELETE), METADATA$ISUPDATE, METADATA$ROW_ID


---

## Tasks: doing it on a schedule, and only when there is work

```sql
CREATE OR REPLACE TASK <name>
    WAREHOUSE = <wh>                 -- or USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE for serverless
    SCHEDULE  = '10 MINUTE'          -- or 'USING CRON <expr> <time_zone>'
    WHEN SYSTEM$STREAM_HAS_DATA('<stream>')
AS
<the statement>;
```

Two details decide whether this works in practice.

**Tasks are created suspended.** Newly created or cloned tasks do not run until you
`ALTER TASK ... RESUME`, and forgetting that is the classic reason a pipeline "silently does nothing".
The role also needs the `EXECUTE TASK` privilege on the account to run any task it owns — revoke it and
every subsequent run stops starting.

**The `WHEN` clause is a cost control, not a nicety.** When a task is triggered by its schedule, it
evaluates the condition first. `SYSTEM$STREAM_HAS_DATA` returning false means the run is skipped and the
warehouse is never started. A ten-minute task over a stream that is usually empty costs almost nothing;
the same task without the `WHEN` clause pays warehouse startup 144 times a day to discover there is
nothing to do.

`EXECUTE TASK <name>` runs a task once, immediately, regardless of its schedule — which is how you test
one without waiting.

→ [More on CREATE TASK](https://docs.snowflake.com/en/sql-reference/sql/create-task)


In [ ]:
%%sql
-- Automate with a TASK
CREATE OR REPLACE TASK GENAI_STUDY.PUBLIC.ENRICH_NEW_TICKETS
    WAREHOUSE = COMPUTE_WH
    SCHEDULE  = '10 MINUTE'
    WHEN SYSTEM$STREAM_HAS_DATA('GENAI_STUDY.PUBLIC.SUPPORT_TICKETS_STREAM')
AS
INSERT INTO GENAI_STUDY.PUBLIC.TICKETS_ENRICHED
    (ticket_id, created_at, category, safe_text, predicted_cat, sentiment, english_text, summary, embedding)
SELECT
    s.ticket_id, s.created_at, s.category,
    AI_REDACT(s.ticket_text),
    AI_CLASSIFY(s.ticket_text, ['billing','technical','shipping','feedback']):labels[0]::VARCHAR,
    (SELECT c.value:sentiment::VARCHAR
       FROM LATERAL FLATTEN(AI_SENTIMENT(s.ticket_text):categories) c
      WHERE c.value:name::VARCHAR = 'overall'),
    CASE WHEN s.language = 'en' THEN s.ticket_text ELSE AI_TRANSLATE(s.ticket_text, '', 'en') END,
    SNOWFLAKE.CORTEX.SUMMARIZE(s.ticket_text),
    AI_EMBED('snowflake-arctic-embed-m-v1.5', s.ticket_text)
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS_STREAM s
WHERE s.METADATA$ACTION = 'INSERT'
  AND s.METADATA$ISUPDATE = FALSE;

-- Tasks are created suspended; nothing runs until this statement.
ALTER TASK GENAI_STUDY.PUBLIC.ENRICH_NEW_TICKETS RESUME;


In [ ]:
%%sql -r task_definition_2
SHOW TASKS LIKE 'ENRICH_NEW_TICKETS' IN SCHEMA GENAI_STUDY.PUBLIC;

-- The task's owner role needs USE AI FUNCTIONS + CORTEX_USER, and EXECUTE TASK on the account.


> ### 🤔 Stop and think
>
> - A ten-minute task gives near-real-time enrichment and pays the AI cost in small, frequent batches. A
>   nightly task batches the same rows into one run. Which is cheaper, which is more useful, and how
>   would you find out what "useful" is worth to the people waiting for the data?
> - The pipeline stores redacted text and throws the original away at the boundary. What happens the day
>   someone needs to reprocess history with a better model — and what would you have to keep for that to
>   be possible without breaking the compliance promise?
> - Every AI-derived column in the target table is a model's opinion frozen at a point in time. When the
>   model changes, those columns silently become a mixture of two models' judgements. How would you notice,
>   and would you backfill?


In [ ]:
%%sql
-- Insert a new ticket to trigger the stream
INSERT INTO GENAI_STUDY.PUBLIC.SUPPORT_TICKETS
    (ticket_id, created_at, customer_name, email, phone, product_id,
     language, category, status, priority, ticket_text)
VALUES
    (9001, CURRENT_TIMESTAMP(), 'Test User', 'test@example.com', '+1-555-9999',
     204, 'en', 'technical', 'open', 'medium',
     'Pipeline test ticket: The new AI enrichment feature is not processing old records. Please investigate.');


In [ ]:
%%sql -r pipeline_test_2
-- Check the stream captured it
SELECT METADATA$ACTION, ticket_id, ticket_text
FROM GENAI_STUDY.PUBLIC.SUPPORT_TICKETS_STREAM;


In [ ]:
%%sql
-- Run the task once, immediately, without waiting for its schedule
EXECUTE TASK GENAI_STUDY.PUBLIC.ENRICH_NEW_TICKETS;


In [ ]:
%%sql -r pipeline_test_4
-- Confirm it landed in the enriched table
SELECT ticket_id, predicted_cat, sentiment, summary
FROM GENAI_STUDY.PUBLIC.TICKETS_ENRICHED
WHERE ticket_id = 9001;


---

## Putting it together

**Scenario.** A healthcare company receives patient feedback forms daily as CSV uploads to a stage.
Compliance needs a table holding only redacted text, with AI-derived sentiment and topic, refreshed within
fifteen minutes of new uploads.

**Which objects do you need, and in what order?**

### Worked solution

**Objects**

1. **Stage** — internal or external, for the CSV uploads
2. **Source table** `RAW_FEEDBACK` — loaded by `COPY INTO` or Snowpipe, with `CHANGE_TRACKING = TRUE`
3. **Stream** — `CREATE STREAM feedback_stream ON TABLE RAW_FEEDBACK`
4. **Target table** `FEEDBACK_COMPLIANT` — `redacted_text`, `topic`, `sentiment`, `processed_at`
5. **Task** — `SCHEDULE = '15 MINUTE'`, `WHEN SYSTEM$STREAM_HAS_DATA(...)`

**Order:** stage → source table with change tracking → stream → target table → task → `ALTER TASK ... RESUME`

```sql
INSERT INTO FEEDBACK_COMPLIANT
SELECT
    AI_REDACT(feedback_text),
    AI_CLASSIFY(feedback_text, ['care','billing','facilities','staff']):labels[0]::VARCHAR,
    (SELECT c.value:sentiment::VARCHAR
       FROM LATERAL FLATTEN(AI_SENTIMENT(feedback_text):categories) c
      WHERE c.value:name::VARCHAR = 'overall'),
    CURRENT_TIMESTAMP()
FROM feedback_stream
WHERE METADATA$ACTION = 'INSERT' AND METADATA$ISUPDATE = FALSE;
```

**Points worth carrying into the exam**

- Tasks are created suspended. `ALTER TASK ... RESUME` is a step, not an afterthought.
- Redact before any other AI call, so personal data never reaches a second model.
- A stream is consumed by the DML that reads it, and a plain `SELECT` does not consume it.
- `WHEN SYSTEM$STREAM_HAS_DATA(...)` means the warehouse does not start when there is nothing to process.
- Filtering on `METADATA$ISUPDATE = FALSE` is what stops updates being re-enriched as if they were new.

**One trade-off to state out loud.** This design writes only redacted text, which is what compliance
asked for. It also means the original wording is gone from this table, so a future model cannot be run
over it and a disputed classification cannot be checked against the source. If that matters, the raw text
has to live somewhere with stricter access control rather than nowhere — a Domain 3 conversation about
masking policies rather than deletion.

→ [More on AI_SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_sentiment)


---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** What are the three stream metadata columns, and what values can `METADATA$ACTION` take?

<details><summary>Show answer</summary>

`METADATA$ACTION` (`INSERT` or `DELETE`), `METADATA$ISUPDATE` (TRUE when the row is one half of an
update), and `METADATA$ROW_ID` (a stable identifier for the row over time). There is no `UPDATE` action —
an update is represented as a paired delete and insert, which is exactly why `METADATA$ISUPDATE` exists.

→ [Streams](https://docs.snowflake.com/en/user-guide/streams-intro)

</details>

**2.** In what state is a newly created task, and which account-level privilege does its owner need to run
it?

<details><summary>Show answer</summary>

Suspended. Newly created or cloned tasks are created suspended and must be resumed with
`ALTER TASK ... RESUME` or invoked manually with `EXECUTE TASK`. The owning role needs the `EXECUTE TASK`
privilege on the account; revoking it stops all subsequent runs from starting under that role.

→ [CREATE TASK](https://docs.snowflake.com/en/sql-reference/sql/create-task)

</details>

**3.** What does `SYSTEM$STREAM_HAS_DATA` do inside a task's `WHEN` clause?

<details><summary>Show answer</summary>

When the schedule fires, the task evaluates the `WHEN` expression before running. If the stream has no
change data, the run is skipped and the warehouse is never started. It is the difference between paying
for warehouse startup on every scheduled tick and paying only when there is work.

→ [CREATE TASK](https://docs.snowflake.com/en/sql-reference/sql/create-task)

</details>

**4.** A pipeline inserts every row from the stream with `WHERE METADATA$ACTION = 'INSERT'` and the target
table fills with duplicates of rows that were only edited. What is missing?

<details><summary>Show answer</summary>

`AND METADATA$ISUPDATE = FALSE`. An update produces a delete/insert pair, so its insert half matches the
action filter. The result is both a duplicate row and a second run of every AI function on text that was
already enriched — the cost shows up before the duplicates are noticed.

→ [Streams](https://docs.snowflake.com/en/user-guide/streams-intro)

</details>

**5.** Someone checks the stream with `SELECT * FROM my_stream;`, sees ten rows, then runs the task and
finds it processed the same ten. Did the `SELECT` lose data?

<details><summary>Show answer</summary>

No — the `SELECT` did nothing to the offset. A stream advances only when it is consumed by a DML
statement, so reading it is safe and repeatable. This is the behaviour you want for debugging. The failure
mode is the mirror image: a pipeline whose "processing" step is a `SELECT` never advances the offset and
reprocesses the same rows on every run.

→ [Streams](https://docs.snowflake.com/en/user-guide/streams-intro)

</details>

**6.** The enrichment query calls `AI_REDACT`, `AI_CLASSIFY`, `AI_SENTIMENT`, `AI_TRANSLATE`,
`SNOWFLAKE.CORTEX.SUMMARIZE` and `AI_EMBED` on each row. For 10,000 new tickets, what have you bought?

<details><summary>Show answer</summary>

Around 60,000 model invocations — one per function per row — with the translate call skipped for rows
already in English by the `CASE` expression. The levers, in order of effect: process only new rows, filter
out rows that do not need a given stage, truncate very long text before it is sent, and drop any derived
column nobody reads.

→ [Cortex AI function cost management](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-func-cost-management)

</details>

**7.** Why does `AI_REDACT` come first in the chain rather than last?

<details><summary>Show answer</summary>

So that no other model ever receives the personal data. If classification ran first, the raw text with
names and e-mail addresses in it would have been sent to a second service before redaction happened —
the final table would look identical, and the exposure would already have occurred. Ordering is the
control here; the column contents do not reveal the difference.

→ [AI_REDACT](https://docs.snowflake.com/en/sql-reference/functions/ai_redact)

</details>

**8.** The target table was originally designed with a `neg_score FLOAT` column populated from
`AI_SENTIMENT`. What happens when the pipeline runs?

<details><summary>Show answer</summary>

The column fills with `NULL`. `AI_SENTIMENT` returns `{"categories":[{"name":..., "sentiment":...}]}` —
labels, with no numeric score anywhere — so whatever path expression was written resolves to nothing and
casts to `NULL`. The column should be a `VARCHAR` holding the overall sentiment label. If a number is
genuinely required, the legacy `SNOWFLAKE.CORTEX.SENTIMENT` returns a FLOAT from −1 to 1.

→ [AI_SENTIMENT](https://docs.snowflake.com/en/sql-reference/functions/ai_sentiment)

</details>

**9.** A task is scheduled every ten minutes and the team wants it every minute for fresher data. What
does that change, and what does it not?

<details><summary>Show answer</summary>

It changes how often the `WHEN` condition is evaluated and how small each batch is; it does not change the
total number of AI calls, since each row is still processed once. What it does cost is warehouse startups
on every tick that has data, and more, smaller warehouse runs are less efficient than fewer large ones.
If the data arrives in a daily upload, a one-minute schedule is 1,439 evaluations that find nothing and
one that does the same work as before.

→ [CREATE TASK](https://docs.snowflake.com/en/sql-reference/sql/create-task)

</details>

**10.** Would you enrich in one large `INSERT ... SELECT` with six AI functions, or split it into stages
that each write an intermediate table? What does each cost?

<details><summary>Show answer</summary>

One statement is cheaper and simpler: the text is read once and every function sees the same row. Staged
tables cost extra storage and extra passes, and buy you restartability — if the embedding step fails on
row 90,000, you have not lost the classification work for the first 89,999. For a small daily batch, one
statement. For a long-running backfill over millions of rows, staging usually pays for itself the first
time something fails.

→ [Cortex AI function cost management](https://docs.snowflake.com/en/user-guide/snowflake-cortex/ai-func-cost-management)

</details>

**11.** Your pipeline must not fail if one malformed row breaks an AI call. What do you add, and what do
you give up?

<details><summary>Show answer</summary>

`return_error_details => TRUE` on the AI functions. Each call then returns `{"value": ..., "error": ...}`
instead of `NULL`, so the batch completes and you can query the failures afterwards. What you give up is
the simple column shape: every downstream reader now has to reach through `:value`, and the target table
column types change. The alternative — leaving it off — is cheaper to write and makes a failure
indistinguishable from an empty answer.

→ [AISQL overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/aisql)

</details>

**12.** *Connecting to another domain.* The enriched table now has an `embedding VECTOR(FLOAT, 768)`
column and the team wants natural-language search over it. What would you build instead, and what does it
require of this pipeline?

<details><summary>Show answer</summary>

A Cortex Search service over the redacted text column, which handles embedding, indexing, hybrid
retrieval and reranking rather than leaving you to maintain vectors by hand. It needs change tracking on
the base objects — which this pipeline already enables for the stream — plus a warehouse and a
`TARGET_LAG` for refresh, and the columns you want to filter on declared as `ATTRIBUTES`. Note you would
then be paying twice if you also keep the `AI_EMBED` column: the service maintains its own embeddings.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>
